[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jobellet/paper2audio/blob/main/notebooks/paper2audio_colab.ipynb)

# paper2audio on a free Colab GPU

Upload a scientific paper (PDF) and get back an MP3 with one chapter per section. The
text is cleaned (citations, statistics, figure references, running heads and captions
removed) and read in the right column order. The voice is
[Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M), a neural TTS model that runs on
the GPU. A 15-page paper takes about 1–2 minutes on a T4.

1. **Runtime → Change runtime type → T4 GPU** (it also works on CPU, just more slowly).
2. **Runtime → Run all**, then pick your PDF when the upload button appears.

If the paper2audio repository is private, add a Colab secret (🔑 in the left sidebar)
named `GITHUB_TOKEN` holding a token that can read it.

In [ ]:
#@title 1. Install (about 1 minute)
REPO = "github.com/jobellet/paper2audio"  #@param {type:"string"}

import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token + '@' if token else ''}{REPO}.git"

!apt-get -qq -y install espeak-ng ffmpeg > /dev/null 2>&1
!pip install -q "kokoro>=0.9.4" soundfile
subprocess.run(["pip", "install", "-q", f"git+{url}"], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (running on CPU)")

In [ ]:
#@title 2. Settings
VOICE = "af_heart"  #@param ["af_heart", "af_bella", "af_nicole", "am_michael", "am_fenrir", "am_puck", "bf_emma", "bf_isabella", "bm_george", "bm_fable"]
SPEED = 1.1  #@param {type:"slider", min:0.7, max:1.6, step:0.05}
READ_CAPTIONS = False  #@param {type:"boolean"}
SKIP_SECTIONS = ""  #@param {type:"string"}
#@markdown `SKIP_SECTIONS`: comma-separated regexes, e.g. `methods, materials`.
PREVIEW = False  #@param {type:"boolean"}
#@markdown `PREVIEW`: only read the title, the abstract and the first section.

In [ ]:
#@title 3. Upload a PDF
from pathlib import Path
from google.colab import files

uploaded = files.upload()
pdf = next(Path(name) for name in uploaded if name.lower().endswith(".pdf"))
print("paper:", pdf)

In [ ]:
#@title 4. Extract and clean the text
from paper2audio import run

options = {}
if READ_CAPTIONS:
    options["read_captions"] = True
if SKIP_SECTIONS.strip():
    options["skip_sections"] = tuple(s.strip() for s in SKIP_SECTIONS.split(",") if s.strip())
script = run(str(pdf), **options)
if PREVIEW:
    script.sections = script.sections[:2]

transcript = pdf.with_suffix(".txt")
transcript.write_text(script.to_text(), encoding="utf-8")
print(f"profile: {script.profile}, year: {script.year or 'unknown'}")
print("sections:", " | ".join(s.title or "(untitled)" for s in script.sections))
if script.skipped:
    print("skipped: ", " | ".join(script.skipped))
print(f"{len(script.to_text().split())} words; transcript saved to {transcript}")

In [ ]:
#@title 5. Synthesise the audio and download it
import re, time
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from kokoro import KPipeline
from paper2audio.audio import render
from paper2audio.tts.base import TTSBackend


class KokoroGPU(TTSBackend):
    """Kokoro-82M (PyTorch) on the GPU; voices starting with 'b' are British English."""
    name = "kokoro-gpu"
    extension = ".wav"

    def __init__(self, voice="af_heart", rate=1.0):
        super().__init__(voice, rate)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.pipeline = KPipeline(lang_code=voice[0], repo_id="hexgrad/Kokoro-82M", device=device)

    def synthesize(self, text, out_path):
        out_path = out_path.with_suffix(self.extension)
        pause = np.zeros(int(24000 * 0.45), dtype="float32")
        chunks = []
        for _, _, audio in self.pipeline(text, voice=self.voice, speed=self.rate, split_pattern=r"\n\s*\n"):
            if audio is not None:
                chunks += [np.asarray(audio.cpu() if hasattr(audio, "cpu") else audio, dtype="float32"), pause]
        sf.write(out_path, np.concatenate(chunks) if chunks else pause, 24000)
        return out_path


start = time.time()
mp3 = render(script, KokoroGPU(VOICE, SPEED), pdf.with_suffix(".mp3"))
print(f"done in {time.time() - start:.0f} s: {mp3}")
display(Audio(str(mp3)))
files.download(str(mp3))
files.download(str(transcript))